# Reverse visual inspection near 19:23:28.22
This is an **exploratory** full-scene review anchored to a known acoustic transient. It cannot identify the sound source or define a biological event. Tile differences can reflect lighting, boundaries, movement, or several seeds. Human interpretation fields in the candidate CSV begin blank. Raw images remain read-only.


In [ ]:
from pathlib import Path
import sys
PROJECT=Path.cwd().resolve()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
if not (PROJECT/'src/reverse_visual.py').exists(): raise RuntimeError('Open inside seed-germination')
sys.path.insert(0,str(PROJECT))
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import Image, Markdown, clear_output, display
from src.reverse_visual import append_manual_candidate, crop_timeline, read_rgb
TABLES=PROJECT/'results/tables/targeted_multimodal'
FIGURES=PROJECT/'results/figures/targeted_multimodal/reverse_visual'
PRIMARY=pd.read_csv(TABLES/'reverse_visual_primary_inventory.csv',parse_dates=['timestamp'])
WIDE=pd.read_csv(TABLES/'reverse_visual_wide_inventory.csv',parse_dates=['timestamp'])
TILES=pd.read_csv(TABLES/'reverse_visual_tile_change_summary.csv')
CANDIDATES=TABLES/'reverse_visual_candidate_regions.csv'
print('Primary frames:',len(PRIMARY),'| wide frames:',len(WIDE))
print('Primary GIF:',FIGURES/'reverse_primary_fullframe.gif')
print('Wide GIF:',FIGURES/'reverse_wide_fullframe.gif')
print('Difference sheet:',FIGURES/'differences/absolute_difference_contact_sheet.png')
display(Image(filename=FIGURES/'reverse_primary_contact_sheet.png'))
display(Image(filename=FIGURES/'top_tiles_overlay.png'))
display(TILES.head(5))
print('Seed_05 comparison:',FIGURES/'seed05_comparison.png')


## Inspect tile timelines
Choose a ranked tile to view all 12 primary-window crops. The score is only a navigation aid. Compare the seed_05 ROI timeline as well.


In [ ]:
tile=widgets.Dropdown(options=[(f'#{rank} {name}',rank) for rank,name in enumerate(TILES.head(5).tile_id,1)],description='Tile:')
tile_output=widgets.Output()
def show_tile(*_):
    with tile_output:
        clear_output(wait=True)
        display(Image(filename=FIGURES/f'top_tile_timelines/top_tile_{tile.value:02d}_timeline.png'))
tile.observe(show_tile,names='value')
display(tile,tile_output);show_tile()
display(Markdown('**Seed_05 ROI timeline for comparison:**'))
display(Image(filename=FIGURES/'seed05_comparison.png'))


## Add a manual candidate ROI
Inspect the representative full frame using pixel coordinates (`x` increases right; `y` increases down). Enter a rectangle around any region you want to review. Saving appends a new `manual_roi` row and creates a timestamped backup of the existing candidate CSV. It does **not** fill any biological interpretation columns.


In [ ]:
representative=read_rgb(FIGURES/'representative_fullframe.png')
height,width=representative.shape[:2]
fig,ax=plt.subplots(figsize=(13,7))
ax.imshow(representative);ax.set_title('Representative full frame — use pixel coordinates to define a manual ROI')
ax.set_xlim(0,width);ax.set_ylim(height,0);ax.grid(alpha=.2);display(fig);plt.close(fig)
fields={name:widgets.IntText(description=name,value=0) for name in ('x_min','y_min','x_max','y_max')}
button=widgets.Button(description='Append manual ROI',button_style='success')
status=widgets.Output()
def save_roi(_):
    with status:
        clear_output(wait=True)
        try:
            bounds=tuple(fields[name].value for name in ('x_min','y_min','x_max','y_max'))
            candidate_id=append_manual_candidate(CANDIDATES,bounds,(width,height))
            row=dict(zip(('x_min','y_min','x_max','y_max'),bounds))
            images=[read_rgb(path) for path in PRIMARY.path]
            output=FIGURES/'manual_candidate_timelines'/f'{candidate_id}_timeline.png'
            crop_timeline(images,PRIMARY.timestamp.tolist(),row,output,candidate_id)
            print('Saved',candidate_id,'and',output)
            display(Image(filename=output))
        except Exception as error:print(type(error).__name__+': '+str(error))
button.on_click(save_roi)
display(widgets.HBox(list(fields.values())),button,status)
print('Candidate review table:',CANDIDATES)
